In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the attack and decay. The top panel is the envelope with its three
control points in gold, and the bottom panel is the 220 Hz tone multiplied
by it. The audio card plays the current shape. Pull the attack all the way
down and you will hear the click that a sudden start produces.

In [ ]:
# hide
# autorun
A0, D0, F0 = 0.1, 0.9, 220.0        # the chapter's adenv(0.1, 0.9) settings

T_TOT = 2.0                         # two seconds on screen
t = np.linspace(0.0, T_TOT, 8000)
TONE = np.sin(2 * np.pi * F0 * t)
SR = 44100
T_PLAY = np.arange(int(T_TOT * SR)) / SR
TONE_P = np.sin(2 * np.pi * F0 * T_PLAY)

def adenv(a_dur, d_dur, tt):
    # the chapter's piecewise-linear attack/decay, via np.interp
    return np.interp(tt, [0.0, a_dur, a_dur + d_dur], [0.0, 1.0, 0.0])

def figure():
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.42, 0.58], vertical_spacing=0.1)
    env = adenv(A0, D0, t)
    fig.add_scatter(x=t * 1000, y=env, mode="lines",
                    line=dict(color=RED, width=2.2), row=1, col=1)
    fig.add_scatter(x=[0, A0 * 1000, (A0 + D0) * 1000], y=[0, 1, 0],
                    mode="markers", marker=dict(color=GOLD, size=9),
                    row=1, col=1)
    fig.add_scatter(x=t * 1000, y=TONE * env, mode="lines",
                    line=dict(color=RED, width=1.0), row=2, col=1)
    for sign in (1.0, -1.0):      # the outline sits on top of the band
        fig.add_scatter(x=t * 1000, y=sign * env, mode="lines",
                        line=dict(color=IRON, width=1.6, dash="dash"),
                        row=2, col=1)
    fig.update_yaxes(range=[-0.08, 1.14], title_text="Envelope",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], title_text="Amplitude",
                     fixedrange=True, row=2, col=1)
    fig.update_xaxes(fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, T_TOT * 1000], title_text="Time (ms)",
                     fixedrange=True, row=2, col=1)
    return fig

def controls(fig):
    a = widgets.FloatSlider(description=r"Attack $a_\text{dur}$ (s)", min=0.005,
                            max=0.5, value=A0, step=0.005, readout_format=".3f")
    d = widgets.FloatSlider(description=r"Decay $d_\text{dur}$ (s)", min=0.05, max=1.5,
                            value=D0, step=0.05)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(a, d, t=t, TONE=TONE, adenv=adenv, readout=readout):
        env = adenv(a, d, t)
        with fig.batch_update():
            fig.data[0].y = env
            fig.data[1].x = [0, a * 1000, (a + d) * 1000]
            fig.data[2].y = TONE * env
            fig.data[3].y = env
            fig.data[4].y = -env
        readout.value = (f"<span style='font-size:0.9em'>control points "
                         f"(0, 0), ({a:.3f}, 1), ({a + d:.3f}, 0) "
                         f"&nbsp;·&nbsp; the note lasts <i>a</i><sub>dur</sub> + "
                         f"<i>d</i><sub>dur</sub> = {a + d:.3f} s</span>")

    widgets.interactive_output(update, {"a": a, "d": d})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(T_PLAY=T_PLAY, TONE_P=TONE_P, SR=SR, adenv=adenv):
        x = 0.125 * TONE_P * adenv(a.value, d.value, T_PLAY)
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)


    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (a, d):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([a, d, readout, out, gate])

icm_plotly.show(figure, controls)